# DSFB-DSCD threshold sweep replay

Generate a run directory from the workspace root with:

```bash
cargo run --release -p dsfb-dscd -- --num-events 8192 --tau-steps 201
```

By default, the next cell auto-detects the DSFB workspace root and selects the newest `output-dsfb-dscd/<timestamp>/` folder.
Set `RUN_DIR` there to an explicit path if you want to pin a specific run.



In [ ]:
from pathlib import Path


def _is_workspace_root(path: Path) -> bool:
    return (path / "Cargo.toml").exists() and (path / "crates" / "dsfb-dscd").exists()


def _discover_workspace_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if _is_workspace_root(candidate):
            return candidate

    search_roots = [Path.home(), Path("/content"), Path("/workspace"), Path("/workspaces")]
    for root in search_roots:
        if not root.exists():
            continue

        if _is_workspace_root(root):
            return root

        for pattern in ("*/Cargo.toml", "*/*/Cargo.toml", "*/*/*/Cargo.toml"):
            for cargo_toml in root.glob(pattern):
                candidate = cargo_toml.parent
                if _is_workspace_root(candidate):
                    return candidate

    raise FileNotFoundError(
        "Could not locate DSFB workspace root (expected Cargo.toml + crates/dsfb-dscd). "
        "Set RUN_DIR explicitly to your output-dsfb-dscd/<timestamp> folder."
    )


WORKSPACE_ROOT = _discover_workspace_root()
OUTPUT_ROOT = WORKSPACE_ROOT / "output-dsfb-dscd"

# Set RUN_DIR to an explicit run folder to pin a specific output.
RUN_DIR = None  # e.g. OUTPUT_ROOT / "20260302_000000"

if RUN_DIR is None:
    run_candidates = sorted(path for path in OUTPUT_ROOT.iterdir() if path.is_dir()) if OUTPUT_ROOT.exists() else []
    if not run_candidates:
        raise FileNotFoundError(
            f"No DSCD output folders found under {OUTPUT_ROOT.resolve()}. "
            "Generate one with: cargo run --release -p dsfb-dscd -- --num-events 8192 --tau-steps 201"
        )
    RUN_DIR = run_candidates[-1]
else:
    RUN_DIR = Path(RUN_DIR)
    if not RUN_DIR.is_dir():
        raise FileNotFoundError(f"Configured RUN_DIR does not exist: {RUN_DIR}")

print(f"Workspace root: {WORKSPACE_ROOT}")
print(f"Using RUN_DIR: {RUN_DIR}")
RUN_DIR



In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

threshold_df = pd.read_csv(RUN_DIR / 'threshold_sweep.csv')
threshold_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(threshold_df['tau'], threshold_df['expansion_ratio'], linewidth=2)
ax.set_title('DSCD expansion ratio vs trust threshold')
ax.set_xlabel('tau')
ax.set_ylabel('expansion_ratio')
ax.grid(alpha=0.25)
plt.show()

In [ ]:
finite_size_path = RUN_DIR / 'finite_size_scaling.csv'
if finite_size_path.exists():
    finite_df = pd.read_csv(finite_size_path)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(finite_df['num_events'], finite_df['transition_width'], marker='o')
    axes[0].set_title('Transition width vs N')
    axes[0].set_xlabel('num_events')
    axes[0].set_ylabel('transition_width')
    axes[0].grid(alpha=0.25)

    axes[1].plot(finite_df['num_events'], finite_df['max_derivative'], marker='o')
    axes[1].set_title('Max derivative vs N')
    axes[1].set_xlabel('num_events')
    axes[1].set_ylabel('max_derivative')
    axes[1].grid(alpha=0.25)
    plt.show()
else:
    print('finite_size_scaling.csv not present for this run')

In [ ]:
events_path = RUN_DIR / 'graph_events.csv'
edges_path = RUN_DIR / 'graph_edges.csv'

events_df = pd.read_csv(events_path)
edges_df = pd.read_csv(edges_path)

subset_edges = edges_df.head(128)
subset_nodes = sorted(set(subset_edges['from_event_id']).union(subset_edges['to_event_id']))

graph = nx.DiGraph()
graph.add_nodes_from(subset_nodes)
graph.add_edges_from(subset_edges[['from_event_id', 'to_event_id']].itertuples(index=False, name=None))

plt.figure(figsize=(10, 6))
pos = nx.spring_layout(graph, seed=7)
nx.draw_networkx(graph, pos=pos, node_size=120, with_labels=False, arrows=True)
plt.title('DSCD graph preview')
plt.axis('off')
plt.show()